In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K


VersionError: Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow/core/framework/attr_value.proto: gencode 5.28.3 runtime 5.28.0. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.

In [ ]:
# PATH
TRAIN_IMG_PATH = "dataset/train/images"
TRAIN_MASK_PATH = "dataset/train/mask"
TEST_IMG_PATH  = "dataset/test/images"

# IMAGE SETTING
IMG_HEIGHT = 256
IMG_WIDTH  = 256
IMG_CHANNELS = 3

BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-4


In [ ]:
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)

    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


In [ ]:
def load_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = img / 255.0
    return img.astype(np.float32)

def load_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 128).astype(np.float32)
    return np.expand_dims(mask, axis=-1)


In [ ]:
X_train, Y_train = [], []

image_files = sorted(os.listdir(TRAIN_IMG_PATH))

for fname in image_files:
    if not fname.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    img_path  = os.path.join(TRAIN_IMG_PATH, fname)
    mask_path = os.path.join(TRAIN_MASK_PATH, fname)

    if not os.path.exists(mask_path):
        continue

    X_train.append(load_image(img_path))
    Y_train.append(load_mask(mask_path))

X_train = np.array(X_train)
Y_train = np.array(Y_train)

print("Train shape:", X_train.shape, Y_train.shape)


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomContrast(0.1),
])


In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return x


In [ ]:
inputs = layers.Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))

x = data_augmentation(inputs)

# Encoder
c1 = conv_block(x, 16)
p1 = layers.MaxPooling2D()(c1)

c2 = conv_block(p1, 32)
p2 = layers.MaxPooling2D()(c2)

c3 = conv_block(p2, 64)
p3 = layers.MaxPooling2D()(c3)

c4 = conv_block(p3, 128)
p4 = layers.MaxPooling2D()(c4)

# Bottleneck
c5 = conv_block(p4, 256)

# Decoder
u6 = layers.UpSampling2D()(c5)
u6 = layers.Concatenate()([u6, c4])
c6 = conv_block(u6, 128)

u7 = layers.UpSampling2D()(c6)
u7 = layers.Concatenate()([u7, c3])
c7 = conv_block(u7, 64)

u8 = layers.UpSampling2D()(c7)
u8 = layers.Concatenate()([u8, c2])
c8 = conv_block(u8, 32)

u9 = layers.UpSampling2D()(c8)
u9 = layers.Concatenate()([u9, c1])
c9 = conv_block(u9, 16)

outputs = layers.Conv2D(1, 1, activation="sigmoid")(c9)

model = Model(inputs, outputs)


In [ ]:
model.compile(
    optimizer=Adam(LR),
    loss=dice_loss,
    metrics=[dice_coef]
)

model.summary()


In [ ]:
history = model.fit(
    X_train, Y_train,
    validation_split=0.1,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True
)


In [ ]:
def predict_mask(img):
    pred = model.predict(img[np.newaxis, ...], verbose=0)[0]
    return pred

THRESHOLD = 0.4


In [ ]:
def encode_rle(mask, pos_value=255):
    binary = (mask == pos_value).astype(np.uint8)
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)


In [ ]:
submission = []

test_files = sorted(os.listdir(TEST_IMG_PATH))

for fname in test_files:
    if not fname.lower().endswith(('.jpg', '.png')):
        continue

    path = os.path.join(TEST_IMG_PATH, fname)
    img = cv2.imread(path)
    h, w = img.shape[:2]

    img_resized = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img_resized = img_resized / 255.0

    pred = predict_mask(img_resized)
    pred = cv2.resize(pred, (w, h))

    mask = np.where(pred > THRESHOLD, 255, 0).astype(np.uint8)

    rle = encode_rle(mask) if mask.max() == 255 else ""
    submission.append([fname, rle])

df = pd.DataFrame(submission, columns=["ImageId", "rle"])
df.to_csv("submission.csv", index=False)
